## Performing Inference
Having produced the best model, we will now use it for inference to produce our results for the `dev` and `test` datasets.

In [1]:
import numpy as np 
import pandas as pd 
from datasets import Dataset
from transformers import DataCollatorWithPadding

/home/azureuser/60035_nlp_cw/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import set_seed

SEED = 42
set_seed(SEED)

MODEL = "roberta-base"
BEST_MODEL_PATH = "../BestModel/best_model"

model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_PATH)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1011.05it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             


In [3]:
'''All preprocessing is done from the raw dataset again here, as we need to 
   preserve all rows / ensure we have 1 prediction per row in the dev set.'''

# Loading in raw dataset for validation set inference 
DATA_PATH = "../data"
DATASET_PATH = f"{DATA_PATH}/dontpatronizeme_pcl.tsv"
VAL_PATH = f"{DATA_PATH}/dev_semeval_parids-labels.csv"
TEST_PATH = f"{DATA_PATH}/task4_test.tsv"

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Load raw data in
raw_data = pd.read_csv(DATASET_PATH, sep="\t", header=None)
raw_data.columns = ["id", "doc_id", "keyword", "country_code", "text", "label"]
data = raw_data[["id", "text", "label"]]

# Fill in null text values within table
data["text"] = data["text"].fillna("")

# Set the "id" of the paragraphs to be the index of the dataframe
data_by_id = data.set_index("id")
data_ids = set(data_by_id.index)


# Loading in dev dataset for inference 
val_df = pd.read_csv(VAL_PATH)
val_ids = val_df["par_id"].tolist()

# Select rows in the exact order of ids
val_data = (
    data_by_id
    .loc[val_ids, ["text", "label"]]
    .reset_index()   # restores "id" as a column
)
# NOTE: Preserve label 2 here, to ensure we retain all rows for dev set
label_map = {
    0: 0,  
    1: 0,
    2: 1,
    3: 1,
    4: 1
}

val_data["label"] = val_data["label"].map(label_map)

assert((val_data["id"] == val_ids).all())

val_data.to_csv("../data/raw_val_data.csv")

val_dataset = Dataset.from_pandas(val_data)
val_dataset = val_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 2094/2094 [00:00<00:00, 16494.06 examples/s]


In [4]:
test_df = pd.read_csv(TEST_PATH, sep="\t", header=None)
test_df = test_df.iloc[:, [-1]] # Retain only the "text" column
test_df.columns = ["text"]

# Construct the `test` dataset
test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 3832/3832 [00:00<00:00, 17028.27 examples/s]


In [5]:
# Predict labels using the model and best threshold identified during training
def predict_labels(trainer, dataset, threshold):
    pred_out = trainer.predict(dataset)
    logits = np.asarray(pred_out.predictions).reshape(-1)
    probs = 1.0 / (1.0 + np.exp(-logits))
    y_pred = (probs >= threshold).astype(int)

    y_true = None
    if pred_out.label_ids is not None:
        y_true = np.asarray(pred_out.label_ids).reshape(-1).astype(int)

    return y_pred, probs, y_true

def save_labels_txt(labels: np.ndarray, path: str):
    labels = np.asarray(labels).reshape(-1)
    with open(path, "w") as f:
        for x in labels:
            f.write(f"{int(x)}\n")

In [6]:
# Load in best model & perform inference on `dev` and `test` sets 
from transformers import TrainingArguments
from transformers.trainer import Trainer

# Pads inputs to the longest sequence in batch. Required to ensure consistent batch sizes
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

args = TrainingArguments(
    per_device_eval_batch_size=16, 
    dataloader_drop_last=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=data_collator,
)

BEST_THRESHOLD = model.config.best_threshold

In [7]:
# Perform inference for test set
from sklearn.metrics import f1_score 

val_pred_labels, val_probs, val_true_labels = predict_labels(trainer, val_dataset, BEST_THRESHOLD)

print(f"Confirming validation F1 score: {f1_score(val_true_labels, val_pred_labels)}")

# Save predicted labels to txt
VAL_SAVE = "dev.txt"
save_labels_txt(val_pred_labels, VAL_SAVE)
print(f"Wrote dev labels to {VAL_SAVE}")


Confirming validation F1 score: 0.5864661654135338
Wrote dev labels to dev.txt


In [8]:
# Perform inference for test set
test_pred_labels, test_probs, _ = predict_labels(trainer, test_dataset, BEST_THRESHOLD)

# Save predicted labels to txt
TEST_SAVE = "test.txt"
save_labels_txt(test_pred_labels, TEST_SAVE)
print(f"Wrote test labels to {TEST_SAVE}")

Wrote test labels to test.txt
